In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
train_df = pd.read_csv("/content/drive/MyDrive/DrugRadar/data/raw/train.csv")
test_df  = pd.read_csv("/content/drive/MyDrive/DrugRadar/data/raw/test.csv")
print(f"Train: {len(train_df)} | Test: {len(test_df)}")

Mounted at /content/drive
Train: 18812 | Test: 4704


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

vec = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_train = vec.fit_transform(train_df['text'])
X_test  = vec.transform(test_df['text'])

clf = LogisticRegression(class_weight='balanced', max_iter=1000)
clf.fit(X_train, train_df['label'])
preds = clf.predict(X_test)

print("── TF-IDF + Logistic Regression ──")
print(classification_report(test_df['label'], preds, target_names=['No ADE', 'ADE']))

── TF-IDF + Logistic Regression ──
              precision    recall  f1-score   support

      No ADE       0.94      0.87      0.90      3340
         ADE       0.73      0.86      0.79      1364

    accuracy                           0.87      4704
   macro avg       0.84      0.87      0.85      4704
weighted avg       0.88      0.87      0.87      4704



In [3]:
import json, os
from sklearn.metrics import f1_score

macro  = f1_score(test_df['label'], preds, average='macro')
ade_f1 = f1_score(test_df['label'], preds, pos_label=1, average='binary')

result = {
    "tfidf_macro_f1":      round(macro, 4),
    "tfidf_ade_f1":        round(ade_f1, 4),
    "xlmroberta_macro_f1": 0.94,
    "xlmroberta_ade_f1":   0.92,
    "improvement_pct":     round((0.94 - macro) / macro * 100, 1)
}
print(result)

os.makedirs("/content/drive/MyDrive/DrugRadar/outputs", exist_ok=True)
with open("/content/drive/MyDrive/DrugRadar/outputs/model_comparison.json", "w") as f:
    json.dump(result, f)
print("✅ Saved!")

{'tfidf_macro_f1': 0.8488, 'tfidf_ade_f1': 0.793, 'xlmroberta_macro_f1': 0.94, 'xlmroberta_ade_f1': 0.92, 'improvement_pct': 10.7}
✅ Saved!
